In [247]:
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple
import json, random
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, Subset

from torch_geometric.data import Data
from torch_geometric.loader import DataLoader
from torch_geometric.nn import GCNConv

In [248]:
# --- Device (CPU/CUDA/DirectML) ---
USE_DIRECTML = False

device = torch.device("cpu")
if not USE_DIRECTML and torch.cuda.is_available():
    device = torch.device("cuda")
elif USE_DIRECTML:
    try:
        import torch_directml
        device = torch_directml.device()
        print("Using DirectML:", device)
    except Exception as e:
        print("DirectML not available, falling back to CPU:", e)
        device = torch.device("cpu")

print("Device:", device)


Device: cpu


In [249]:
# --- Token utils ---
NUC2ID = {'A':1, 'C':2, 'G':3, 'T':4, 'N':5}
ID2NUC = {v:k for k,v in NUC2ID.items()}

def seq_to_tokens(seq: str) -> torch.LongTensor:
    return torch.tensor([NUC2ID.get(c, 5) for c in seq], dtype=torch.long)

def tokens_to_onehot(tokens: torch.LongTensor, num_classes: int = 6):
    # tokens: [N,K]
    N, K = tokens.shape
    classes = torch.arange(num_classes, device=tokens.device).view(1, 1, num_classes)
    return (tokens.unsqueeze(-1) == classes).float()  # [N,K,C]

def _safe_get(d: Dict[str, Any], key: str, default):
    v = d.get(key, default)
    return default if v is None else v


In [250]:
class HyperbubbleDataset(Dataset):
    """
    Minimal GNN-ready dataset with token sequences for CNN encoding.
    Ładuje JSONL z rekordami bąbli.
    """
    def __init__(self, files: List[str]):
        self.files = [Path(p) for p in files]
        self.records: List[Dict[str, Any]] = []
        for fp in self.files:
            with fp.open("r") as fh:
                for line in fh:
                    line = line.strip()
                    if not line:
                        continue
                    try:
                        self.records.append(json.loads(line))
                    except json.JSONDecodeError:
                        continue

    def __len__(self) -> int:
        return len(self.records)

    def _build_graph(self, rec: Dict[str, Any]) -> Data:
        node_seqs: Dict[str, int] = {}
        node_covs: List[float] = []

        def ensure_node(seq: str, cov: Optional[float]=None) -> int:
            if seq not in node_seqs:
                node_seqs[seq] = len(node_seqs)
                node_covs.append(float(cov) if cov is not None else 0.0)
            else:
                i = node_seqs[seq]
                if cov is not None and node_covs[i] == 0:
                    node_covs[i] = float(cov)
            return node_seqs[seq]

        start_seq = rec["start_seq"]
        end_seq   = rec["end_seq"]

        for n in _safe_get(rec,"nodes",[]): ensure_node(n["seq"], n.get("cov"))
        for n in _safe_get(rec,"upstream_nodes",[]): ensure_node(n["seq"], n.get("cov"))
        for n in _safe_get(rec,"downstream_nodes",[]): ensure_node(n["seq"], n.get("cov"))
        for n in _safe_get(rec,"downstream_nodes",[]): ensure_node(n["seq"], n.get("cov"))
        for n in _safe_get(rec,"inside_nodes",[]):
            if isinstance(n,dict) and "seq" in n:
                ensure_node(n["seq"], n.get("cov"))

        ensure_node(start_seq)
        ensure_node(end_seq)

        edge_src, edge_dst, edge_attr = [], [], []
        for e in _safe_get(rec,"edges",[]):
            u = ensure_node(e["source_seq"])
            v = ensure_node(e["target_seq"])
            edge_src.append(u)
            edge_dst.append(v)
            edge_attr.append([
                float(e.get("len_nodes",0)),
                float(e.get("len_bp",0)),
                float(e.get("cov_min",0)),
                float(e.get("cov_mean",0.0)),
                1.0 if e.get("on_ref",False) else 0.0
            ])

        node_order = [None]*len(node_seqs)
        for s,i in node_seqs.items():
            node_order[i]=s

        seq_tokens = torch.stack([seq_to_tokens(s) for s in node_order], dim=0)  # [N,K]
        x_cov = torch.tensor(node_covs, dtype=torch.float32).unsqueeze(1)        # [N,1]

        start_idx = torch.tensor(node_seqs[start_seq])
        end_idx   = torch.tensor(node_seqs[end_seq])

        edge_index = torch.tensor([edge_src, edge_dst], dtype=torch.long)
        edge_attr  = torch.tensor(edge_attr, dtype=torch.float32) if edge_attr else torch.zeros((0,5))

        num_edges = edge_index.size(1)
        y_edge_mask = torch.zeros(num_edges, dtype=torch.long)
        label_path_idx = -1

        paths_list = _safe_get(rec,"paths",[])
        lp = rec.get("label_path")
        if lp is not None and 0 <= lp < len(paths_list):
            label_path_idx = lp
            if num_edges > 0:
                y_edge_mask[torch.tensor(paths_list[lp], dtype=torch.long)] = 1

        data = Data(
            seq_tokens=seq_tokens,
            x_cov=x_cov,
            edge_index=edge_index,
            edge_attr=edge_attr,
            start_idx=start_idx,
            end_idx=end_idx,
            num_nodes=seq_tokens.size(0),
            y_edge_mask=y_edge_mask,
            label_path_idx=torch.tensor(label_path_idx),
        )
        if "bubble_id" in rec:
            data.bubble_id = torch.tensor(rec["bubble_id"])
        if "k" in rec:
            data.k = torch.tensor(rec["k"])
        return data

    def __getitem__(self, idx: int) -> Data:
        return self._build_graph(self.records[idx])


def labeled_subset(ds: Dataset) -> Subset:
    labeled_indices = []
    for i in range(len(ds)):
        d = ds[i]
        lp = int(d.label_path_idx.item()) if hasattr(d, "label_path_idx") else -1
        if lp >= 0:
            labeled_indices.append(i)
    return Subset(ds, labeled_indices)


In [251]:
class HyperbubbleGNN(nn.Module):
    """
    CNN-based sequence encoder + 2x GCNConv + MLP edge classifier.
    """
    def __init__(
        self,
        vocab_size=6,
        cnn_channels=32,
        gcn_hidden=64,
        edge_hidden=64,
        edge_feat_dim=5,
        dropout=0.0,
    ):
        super().__init__()

        self.cnn = nn.Conv1d(vocab_size, cnn_channels, kernel_size=3, padding=1)
        self.node_in = cnn_channels + 1

        self.gcn1 = GCNConv(self.node_in, gcn_hidden)
        self.gcn2 = GCNConv(gcn_hidden, gcn_hidden)

        self.dropout = nn.Dropout(dropout)

        self.edge_mlp = nn.Sequential(
            nn.Linear(2 * gcn_hidden + edge_feat_dim, edge_hidden),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(edge_hidden, 1),
        )

    def encode_nodes(self, seq_tokens, x_cov):
        onehot = tokens_to_onehot(seq_tokens)   # [N,K,C]
        x = onehot.permute(0, 2, 1)             # [N,C,K]
        H = F.relu(self.cnn(x))                 # [N,C,K]
        H = H.mean(dim=2)                       # [N,C]
        return torch.cat([H, x_cov], dim=1)     # [N,C+1]

    def forward(self, data):
        X0 = self.encode_nodes(data.seq_tokens, data.x_cov)
        H = F.relu(self.gcn1(X0, data.edge_index))
        H = self.dropout(H)
        H = F.relu(self.gcn2(H, data.edge_index))
        H = self.dropout(H)

        u, v = data.edge_index
        U = H[u]
        V = H[v]

        if hasattr(data, "edge_attr") and data.edge_attr.numel() > 0:
            EA = data.edge_attr
        else:
            E = data.edge_index.size(1)
            EA = H.new_zeros((E, 5))

        feats = torch.cat([U, V, EA], dim=1)
        logits = self.edge_mlp(feats).squeeze(-1)
        return logits


In [252]:

DATA_ROOT = Path("other_ecoli_data")

COVERAGE = 20
K = 21
RATIO_TAG = "ratio_lt_3.5"


FILE_GLOB = f"*dataset_k{K}_cov{COVERAGE}_{RATIO_TAG}.jsonl"
print("DATA_ROOT:", DATA_ROOT.resolve())
print("FILE_GLOB:", FILE_GLOB)


DATA_ROOT: C:\Users\Przemek\Desktop\Inzynierka stuff\own_assembler\dna-assembly-bsc\bubble_resolution_gnn\other_ecoli_data
FILE_GLOB: *dataset_k21_cov20_ratio_lt_3.5.jsonl


In [253]:
def discover_genome_files(data_root: Path, file_glob: str) -> Dict[str, str]:
    paths = sorted([p.resolve() for p in data_root.glob(file_glob)])
    if not paths:
        raise FileNotFoundError(f"No files in {data_root} matching: {file_glob}")

    mapping: Dict[str, str] = {}
    for p in paths:
        # genome_id = prefix przed "_dataset_..."
        genome_id = p.name.split("_dataset_")[0]
        mapping[genome_id] = str(p)

    print(f"[discover] found {len(mapping)} genomes:")
    for gid in sorted(mapping.keys()):
        print("  ", gid)
    return mapping

GENOME_TO_PATH = discover_genome_files(DATA_ROOT, FILE_GLOB)
GENOMES = sorted(GENOME_TO_PATH.keys())

[discover] found 3 genomes:
   ecoli_variant1
   ecoli_variant2
   ecoli_variant3


In [254]:
def split_genomes_for_holdout(
    genomes: List[str],
    test_genomes: int,
    val_mode: str,                   # "train_frac" | "separate_genomes"
    val_genomes: int,
    seed: int,
) -> Tuple[List[str], List[str], List[str]]:
    """
    Zwraca: train_genomes, val_genomes_list, test_genomes_list
    - test: zawsze genomy
    - val: zależnie od val_mode
    """
    assert val_mode in ("train_frac", "separate_genomes")
    rng = np.random.default_rng(seed)
    g = np.array(genomes, dtype=object)
    rng.shuffle(g)

    if test_genomes >= len(g):
        raise ValueError("test_genomes must be < number of available genomes")

    test = g[:test_genomes].tolist()
    rest = g[test_genomes:].tolist()

    if val_mode == "train_frac":
        val = []
        train = rest
    else:
        if val_genomes <= 0:
            raise ValueError("val_genomes must be > 0 for separate_genomes mode")
        if val_genomes >= len(rest):
            raise ValueError("val_genomes must be < remaining genomes after test split")
        val = rest[:val_genomes]
        train = rest[val_genomes:]

    return train, val, test


def split_indices_train_val(n: int, val_frac: float, seed: int):
    rng = np.random.default_rng(seed)
    order = np.arange(n)
    rng.shuffle(order)
    n_val = int(round(val_frac * n))
    val_idx = order[:n_val].tolist()
    tr_idx  = order[n_val:].tolist()
    return tr_idx, val_idx


In [255]:
def kfold_genome_splits(
    genomes: List[str],
    k_folds: int,
    seed: int,
    val_mode: str,
    val_genomes: int,
) -> List[Dict[str, List[str]]]:
    """
    Zwraca listę foldów: {"train": [...], "val_genomes": [...], "test": [...]}
    """
    assert k_folds >= 2
    assert val_mode in ("train_frac", "separate_genomes")

    rng = np.random.default_rng(seed)
    g = np.array(genomes, dtype=object)
    rng.shuffle(g)

    folds = np.array_split(g, k_folds)
    out = []

    for i in range(k_folds):
        test = folds[i].tolist()
        rest = np.concatenate([folds[j] for j in range(k_folds) if j != i]).tolist()

        if val_mode == "train_frac":
            val_g = []
            train_g = rest
        else:
            if val_genomes <= 0:
                raise ValueError("val_genomes must be > 0 for separate_genomes mode")
            if val_genomes >= len(rest):
                raise ValueError("val_genomes must be < genomes available in train+val for this fold")
            # deterministycznie: weź pierwsze val_genomes z rest
            val_g = rest[:val_genomes]
            train_g = rest[val_genomes:]

        out.append({"train": train_g, "val_genomes": val_g, "test": test})

    return out

In [256]:
def make_loaders_for_fold(
    genome_to_path: Dict[str, str],
    train_genomes: List[str],
    val_genomes: List[str],
    test_genomes: List[str],
    batch_size: int,
    num_workers: int,
    val_mode: str,     # "train_frac" | "separate_genomes"
    val_frac: float,
    seed: int,
):
    train_paths = [genome_to_path[g] for g in train_genomes]
    test_paths  = [genome_to_path[g] for g in test_genomes]

    train_full = HyperbubbleDataset(train_paths)
    train_lab  = labeled_subset(train_full)

    test_full = HyperbubbleDataset(test_paths)
    test_lab  = labeled_subset(test_full)

    if len(train_lab) == 0:
        raise RuntimeError("No labeled samples in train set.")
    if len(test_lab) == 0:
        raise RuntimeError("No labeled samples in test set.")

    if val_mode == "train_frac":
        tr_idx, va_idx = split_indices_train_val(len(train_lab), val_frac, seed)
        train_ds = Subset(train_lab, tr_idx)
        val_ds   = Subset(train_lab, va_idx)
    else:
        val_paths = [genome_to_path[g] for g in val_genomes]
        val_full  = HyperbubbleDataset(val_paths)
        val_lab   = labeled_subset(val_full)
        if len(val_lab) == 0:
            raise RuntimeError("No labeled samples in val genomes.")
        train_ds = train_lab
        val_ds   = val_lab

    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True,  num_workers=num_workers)
    val_loader   = DataLoader(val_ds,   batch_size=batch_size, shuffle=False, num_workers=num_workers)
    test_loader  = DataLoader(test_lab, batch_size=batch_size, shuffle=False, num_workers=num_workers)

    sizes = {
        "train_bubbles": len(train_ds),
        "val_bubbles": len(val_ds),
        "test_bubbles": len(test_lab),
    }
    return train_loader, val_loader, test_loader, sizes


In [257]:

def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def split_train_val(n: int, val_frac: float, seed: int):
    rng = np.random.default_rng(seed)
    order = np.arange(n)
    rng.shuffle(order)
    n_val = int(round(val_frac * n))
    val_idx = order[:n_val]
    tr_idx  = order[n_val:]
    return tr_idx.tolist(), val_idx.tolist()


def count_params(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


def _num_graphs_in_batch(node_batch: torch.Tensor) -> int:
    return int(node_batch.max().item()) + 1 if node_batch.numel() else 0


def _edge_batch_from_nodes(node_batch: torch.Tensor, edge_src: torch.Tensor) -> torch.Tensor:
    return node_batch[edge_src] if edge_src.numel() else edge_src.new_zeros((0,), dtype=torch.long)


def _collect_decisions_by_labeled_sources(data, g: int, u: torch.Tensor, edge_batch: torch.Tensor):
    """
    Zwraca listę (cand_idx, gold_pos) dla każdego węzła źródłowego w grafie g,
    dla którego istnieje decyzja (zaznaczone krawędzie y_edge_mask==1).
    """
    y = data.y_edge_mask
    lab_eidx = (y == 1).nonzero(as_tuple=False).view(-1)
    lab_eidx = lab_eidx[edge_batch[lab_eidx] == g]
    if lab_eidx.numel() == 0:
        return []

    labeled_sources = torch.unique(u[lab_eidx])

    decisions = []
    for s in labeled_sources.tolist():
        mask = (edge_batch == g) & (u == s)
        cand_idx = mask.nonzero(as_tuple=False).view(-1)
        if cand_idx.numel() < 2:
            continue
        y_here = y[cand_idx]
        gold_pos = (y_here == 1).nonzero(as_tuple=False).view(-1)
        if gold_pos.numel() == 1:
            decisions.append((cand_idx, gold_pos.view(1)))
    return decisions


@torch.no_grad()
def _candidate_sets(data, logits: torch.Tensor):
    """
    Zwraca listę (cand_idx [M], gold_pos [1], g) dla każdej decyzji w batchu.
    """
    u = data.edge_index[0]
    node_batch = (
        data.batch if hasattr(data, "batch")
        else torch.zeros(data.num_nodes, dtype=torch.long, device=logits.device)
    )
    edge_batch = _edge_batch_from_nodes(node_batch, u)
    num_graphs = _num_graphs_in_batch(node_batch)

    out = []
    for g in range(num_graphs):
        decisions = _collect_decisions_by_labeled_sources(data, g, u, edge_batch)
        for cand_idx, gold_pos in decisions:
            out.append((cand_idx, gold_pos, g))
    return out


def decision_ce_loss_from_batch(logits: torch.Tensor, data, reduction: str = "mean"):
    """
    Cross-entropy liczona po decyzjach (candidate sets).
    Zwraca: (loss, liczba_decyzji)
    """
    if logits.numel() == 0:
        return logits.new_zeros(()), 0

    losses, decisions = [], 0
    u = data.edge_index[0]
    node_batch = (
        data.batch if hasattr(data, "batch")
        else torch.zeros(data.num_nodes, dtype=torch.long, device=logits.device)
    )
    edge_batch = _edge_batch_from_nodes(node_batch, u)
    num_graphs = _num_graphs_in_batch(node_batch)

    for g in range(num_graphs):
        cands = _collect_decisions_by_labeled_sources(data, g, u, edge_batch)
        for cand_idx, gold_pos in cands:
            cand_logits = logits[cand_idx]     # [M]
            target = gold_pos.view(1)          # [1]
            losses.append(F.cross_entropy(cand_logits.unsqueeze(0), target))
            decisions += 1

    if decisions == 0:
        return logits.new_zeros(()), 0

    loss = torch.stack(losses)
    return (loss.mean() if reduction == "mean" else loss.sum()), decisions


@torch.no_grad()
def eval_rich_metrics(model, loader, device) -> Dict[str, float]:
    """
    Minimalny zestaw metryk:
      - acc_dec   : dokładność decyzji (wybór ścieżki)
      - acc_bub   : dokładność bąbli (wszystkie decyzje w bąblu poprawne)
      - precision_dec, recall_dec, f1_dec : metryki binarne dla decyzji
      - brier     : błąd Briera dla prawdopodobieństwa poprawnej ścieżki
    """
    model.eval()
    dec_correct, dec_total = 0, 0
    bubble_ok = {}  # key: (id(data), g)

    confs_gold = []
    labels_gold = []

    y_true_dec = []
    y_pred_dec = []

    for data in loader:
        data = data.to(device)
        logits = model(data)
        if logits.numel() == 0:
            continue

        for cand_idx, gold_pos, g in _candidate_sets(data, logits):
            cand_logits = logits[cand_idx]              # [M]
            probs = F.softmax(cand_logits, dim=0)
            pred = int(torch.argmax(cand_logits).item())
            gold = int(gold_pos.item())

            ok = int(pred == gold)
            dec_total += 1
            dec_correct += ok

            key = (id(data), int(g))
            bubble_ok.setdefault(key, True)
            bubble_ok[key] = bubble_ok[key] and bool(ok)

            conf_gold = float(probs[gold].item())
            confs_gold.append(conf_gold)
            labels_gold.append(float(ok))

            y_true_dec.append(float(ok))
            conf_pred = float(probs[pred].item())
            y_pred_dec.append(1.0 if conf_pred >= 0.5 else 0.0)

    acc_dec = dec_correct / max(dec_total, 1)

    bub_total = len(bubble_ok)
    bub_correct = sum(1 for ok in bubble_ok.values() if ok)
    acc_bub = bub_correct / max(bub_total, 1)

    confs_gold = np.array(confs_gold, dtype=np.float64)
    labels_gold = np.array(labels_gold, dtype=np.float64)
    brier = float(((confs_gold - labels_gold) ** 2).mean()) if confs_gold.size else float("nan")

    y_true_dec = np.array(y_true_dec, dtype=np.int32)
    y_pred_dec = np.array(y_pred_dec, dtype=np.int32)
    if y_true_dec.size:
        tp = int(((y_true_dec == 1) & (y_pred_dec == 1)).sum())
        fp = int(((y_true_dec == 0) & (y_pred_dec == 1)).sum())
        fn = int(((y_true_dec == 1) & (y_pred_dec == 0)).sum())

        precision_dec = tp / max(tp + fp, 1)
        recall_dec    = tp / max(tp + fn, 1)
        if precision_dec + recall_dec > 0:
            f1_dec = 2 * precision_dec * recall_dec / (precision_dec + recall_dec)
        else:
            f1_dec = 0.0
    else:
        precision_dec = recall_dec = f1_dec = float("nan")

    return {
        "acc_dec": float(acc_dec),
        "acc_bub": float(acc_bub),
        "precision_dec": float(precision_dec),
        "recall_dec": float(recall_dec),
        "f1_dec": float(f1_dec),
        "brier": float(brier),
        "decisions": int(dec_total),
        "bubbles": int(bub_total),
    }


def train_one_epoch(model, loader, device, optimizer, grad_clip: Optional[float] = 1.0):
    model.train()
    total_loss, total_decisions = 0.0, 0

    for data in loader:
        data = data.to(device)
        optimizer.zero_grad(set_to_none=True)

        logits = model(data)
        loss, ndec = decision_ce_loss_from_batch(logits, data, reduction="mean")
        if ndec == 0:
            continue

        loss.backward()
        if grad_clip is not None:
            torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
        optimizer.step()

        # loss jest mean -> ważymy przez liczbę decyzji
        total_loss += float(loss.item()) * ndec
        total_decisions += ndec

    return total_loss / max(total_decisions, 1)


@torch.no_grad()
def eval_loss_only(model, loader, device) -> Tuple[float, int]:
    """
    Zwraca (średni loss po decyzjach, liczba decyzji) na danym loaderze.
    """
    model.eval()
    total_loss = 0.0
    total_decisions = 0

    for data in loader:
        data = data.to(device)
        logits = model(data)
        loss, ndec = decision_ce_loss_from_batch(logits, data, reduction="mean")
        if ndec == 0:
            continue
        total_loss += float(loss.item()) * ndec
        total_decisions += ndec

    if total_decisions == 0:
        return float("nan"), 0
    return total_loss / total_decisions, total_decisions


def fit_variant(
    build_model_fn,
    train_loader,
    val_loader,
    device,
    epochs=20,
    lr=1e-3,
    weight_decay=0,
    patience=5,
    grad_clip: Optional[float] = 1.0,
    seed=42,
    print_every=5,
    early_stop=True,
    out_ckpt: Optional[Path] = None,
    select_by: str = "val_acc_bub",   # "val_acc_bub" | "train_acc_bub" | "train_loss" | "none"
):
    _ = seed

    model = build_model_fn().to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)

    best_metric = -1e18
    best_state = None
    waited = 0
    history = []

    has_val = (val_loader is not None)

    def _metric_from(train_loss, train_stats, val_loss, val_stats):
        if select_by == "val_acc_bub":
            if not has_val:
                return float("nan")
            return float(val_stats["acc_bub"])
        if select_by == "val_acc_dec":
            if not has_val:
                return float("nan")
            return float(val_stats["acc_dec"])
        if select_by == "train_acc_bub":
            return float(train_stats["acc_bub"])
        if select_by == "train_loss":
            # maximize metric -> use negative loss
            return -float(train_loss)
        if select_by == "none":
            return float("nan")
        raise ValueError(f"Unknown select_by={select_by}")

    for ep in range(1, epochs + 1):
        train_loss = train_one_epoch(model, train_loader, device, opt, grad_clip)
        train_stats = eval_rich_metrics(model, train_loader, device)

        if has_val:
            val_loss, _ = eval_loss_only(model, val_loader, device)
            val_stats = eval_rich_metrics(model, val_loader, device)
        else:
            val_loss = float("nan")
            val_stats = {
                "acc_dec": float("nan"),
                "acc_bub": float("nan"),
                "precision_dec": float("nan"),
                "recall_dec": float("nan"),
                "f1_dec": float("nan"),
                "brier": float("nan"),
                "decisions": 0,
                "bubbles": 0,
            }

        metric = _metric_from(train_loss, train_stats, val_loss, val_stats)

        history.append({
            "epoch": ep,
            "train_loss": train_loss,
            "train_acc_dec": float(train_stats["acc_dec"]),
            "train_acc_bub": float(train_stats["acc_bub"]),
            "val_loss": float(val_loss),
            "val_acc_dec": float(val_stats["acc_dec"]),
            "val_acc_bub": float(val_stats["acc_bub"]),
            "metric_used": float(metric) if metric == metric else float("nan"),
            "select_by": select_by,
        })

        if (ep % print_every == 0) or (ep == 1):
            print(
                f"[ep {ep:03d}] "
                f"train_loss={train_loss:.4f} "
                f"train_acc_dec={train_stats['acc_dec']:.4f} "
                f"train_acc_bub={train_stats['acc_bub']:.4f} "
                f"val_loss={val_loss:.4f} "
                f"val_acc_dec={val_stats['acc_dec']:.4f} "
                f"val_acc_bub={val_stats['acc_bub']:.4f} "
                f"metric_used={(metric if metric==metric else float('nan')):.4f}"
            )

        if select_by == "none":
            # keep last epoch
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            continue

        # model selection
        if metric == metric and metric > best_metric:  # not NaN
            best_metric = metric
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            waited = 0
        else:
            waited += 1

        # early stop policy:
        # - if selecting by val, require val
        # - if selecting by train metric/loss, allow early stop without val
        if early_stop:
            if select_by == "val_acc_bub" and not has_val:
                pass
            else:
                if waited >= patience:
                    break

    if best_state is None:
        best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

    if out_ckpt is not None:
        out_ckpt.parent.mkdir(parents=True, exist_ok=True)
        torch.save(best_state, out_ckpt)

    return best_state, pd.DataFrame(history)


In [258]:
# ============ EXPERIMENT CONFIG ============
CFG = {
    "seed": random.randint(1, 100),
    "batch_size": 64,
    "num_workers": 0,

    "epochs": 200,
    "patience": 200,
    "lr": 1e-3,
    "weight_decay": 1e-4,
    "grad_clip": 1.0,
    "print_every": 25,

    # validation enabled (bubble-level split from train genomes):
    "val_mode": "train_frac",     # "train_frac" | "separate_genomes"
    "val_frac": 0.1,             # <-- restore > 0
    "val_genomes": 0,             # used only if val_mode="separate_genomes"

    # test:
    "kfold_mode": True,
    "k_folds": 6,
    "test_genomes": 2,            # used only when kfold_mode=False

    # model:
    "cnn_channels": 8,
    "gcn_hidden": 16,
    "edge_hidden": 256,
    "dropout": 0.0,

    # selection:
    "select_by": "val_acc_bub",   # <-- select best checkpoint by VAL again
    "early_stop": True,
}

def build_model():
    return HyperbubbleGNN(
        vocab_size=6,
        cnn_channels=CFG["cnn_channels"],
        gcn_hidden=CFG["gcn_hidden"],
        edge_hidden=CFG["edge_hidden"],
        edge_feat_dim=5,
        dropout=CFG["dropout"],
    )

set_seed(CFG["seed"])
print("Genomes available:", len(GENOMES))


Genomes available: 3


In [259]:
from typing import List
import pandas as pd

def _short_list(xs, n=5):
    xs = list(xs)
    if len(xs) <= 2 * n:
        return xs
    return xs[:n] + ["..."] + xs[-n:]


def run_one_fold(
    fold_id: str,
    train_genomes: List[str],
    val_genomes: List[str],
    test_genomes: List[str],
):
    # --- sanity checks ---
    if len(test_genomes) == 0:
        print(f"\n=== FOLD {fold_id} ===")
        print("[SKIP] empty test_genomes (likely k_folds > #genomes).")
        return None, None

    print(f"\n=== FOLD {fold_id} ===")
    print("train genomes:", _short_list(train_genomes))
    print("val genomes  :", _short_list(val_genomes) if val_genomes else "(from train bubbles)")
    print("test genomes :", _short_list(test_genomes))

    # --- build loaders ---
    train_loader, val_loader, test_loader, sizes = make_loaders_for_fold(
        genome_to_path=GENOME_TO_PATH,
        train_genomes=train_genomes,
        val_genomes=val_genomes,
        test_genomes=test_genomes,
        batch_size=CFG["batch_size"],
        num_workers=CFG["num_workers"],
        val_mode=CFG["val_mode"],
        val_frac=CFG["val_frac"],
        seed=CFG["seed"],
    )

    # --- train + model selection ---
    best_state, hist = fit_variant(
        build_model_fn=build_model,
        train_loader=train_loader,
        val_loader=val_loader,
        device=device,
        epochs=CFG["epochs"],
        lr=CFG["lr"],
        weight_decay=CFG["weight_decay"],
        patience=CFG["patience"],
        grad_clip=CFG["grad_clip"],
        seed=CFG.get("seed", 42),
        print_every=CFG["print_every"],
        early_stop=CFG.get("early_stop", True),
        out_ckpt=None,
        select_by=CFG.get("select_by", "val_acc_bub"),
    )

    # --- test eval (ALWAYS) ---
    model = build_model().to(device)
    model.load_state_dict(best_state)
    model.eval()

    test_stats = eval_rich_metrics(model, test_loader, device)

    test_loss = None
    if "eval_loss_only" in globals():
        tl, _ = eval_loss_only(model, test_loader, device)
        test_loss = tl

    params = count_params(model)

    msg = (
        f"[TEST {fold_id}] "
        f"acc_dec={test_stats['acc_dec']:.4f} "
        f"acc_bub={test_stats['acc_bub']:.4f} "
        f"decisions={test_stats['decisions']} "
        f"bubbles={test_stats['bubbles']}"
    )
    if test_loss is not None and test_loss == test_loss:
        msg += f" test_loss={test_loss:.4f}"
    print(msg)

    row = {
        "fold": fold_id,
        "params": params,
        "train_genomes": ",".join(train_genomes),
        "val_genomes": ",".join(val_genomes),
        "test_genomes": ",".join(test_genomes),
        **sizes,
        "test_loss": test_loss,
        "test_acc_dec": test_stats["acc_dec"],
        "test_acc_bub": test_stats["acc_bub"],
        "test_precision_dec": test_stats["precision_dec"],
        "test_recall_dec": test_stats["recall_dec"],
        "test_f1_dec": test_stats["f1_dec"],
        "test_brier": test_stats["brier"],
        "test_decisions": test_stats["decisions"],
        "test_bubbles": test_stats["bubbles"],
    }
    return row, hist


# ==============================
# Multi-seed retry wrapper
# ==============================

N_RETRIES = 5

all_results = []
all_histories = []

base_seed = CFG.get("seed", 42)

for retry in range(1, N_RETRIES + 1):
    print(f"\n==============================")
    print(f" RETRY {retry}/{N_RETRIES}")
    print(f"==============================")

    retry_seed = random.randint(1, 10_000_000)
    CFG["seed"] = retry_seed
    set_seed(retry_seed)

    print(f"[retry {retry}] seed = {retry_seed}")

    results = []
    histories = []

    # --- guard against impossible k-fold ---
    if CFG["kfold_mode"]:
        if CFG["k_folds"] > len(GENOMES):
            print(
                f"[WARN] k_folds={CFG['k_folds']} > #genomes={len(GENOMES)} "
                f"-> setting k_folds={len(GENOMES)}"
            )
            CFG["k_folds"] = len(GENOMES)

    if not CFG["kfold_mode"]:
        tr_g, va_g, te_g = split_genomes_for_holdout(
            genomes=GENOMES,
            test_genomes=CFG["test_genomes"],
            val_mode=CFG["val_mode"],
            val_genomes=CFG["val_genomes"],
            seed=CFG["seed"],
        )
        row, hist = run_one_fold("holdout", tr_g, va_g, te_g)
        if row is not None:
            row["retry"] = retry
            row["seed"] = retry_seed
            results.append(row)
            histories.append(hist)
    else:
        folds = kfold_genome_splits(
            genomes=GENOMES,
            k_folds=CFG["k_folds"],
            seed=CFG["seed"],
            val_mode=CFG["val_mode"],
            val_genomes=CFG["val_genomes"],
        )
        for i, f in enumerate(folds, start=1):
            row, hist = run_one_fold(f"{i}/{CFG['k_folds']}", f["train"], f["val_genomes"], f["test"])
            if row is not None:
                row["retry"] = retry
                row["seed"] = retry_seed
                results.append(row)
                histories.append(hist)

    df_results = pd.DataFrame(results)
    all_results.append(df_results)
    all_histories.append(histories)

# restore original seed
CFG["seed"] = base_seed
set_seed(base_seed)

df_all_results = pd.concat(all_results, ignore_index=True)
df_all_results


 RETRY 1/5
[retry 1] seed = 9391146
[WARN] k_folds=6 > #genomes=3 -> setting k_folds=3

=== FOLD 1/3 ===
train genomes: ['ecoli_variant2', 'ecoli_variant1']
val genomes  : (from train bubbles)
test genomes : ['ecoli_variant3']
[ep 001] train_loss=0.7165 train_acc_dec=0.5442 train_acc_bub=0.4798 val_loss=0.6843 val_acc_dec=0.5570 val_acc_bub=0.5373 metric_used=0.5373
[ep 025] train_loss=0.6704 train_acc_dec=0.5734 train_acc_bub=0.5115 val_loss=0.6854 val_acc_dec=0.4899 val_acc_bub=0.4403 metric_used=0.4403
[ep 050] train_loss=0.6535 train_acc_dec=0.5873 train_acc_bub=0.5088 val_loss=0.7439 val_acc_dec=0.4497 val_acc_bub=0.4179 metric_used=0.4179
[ep 075] train_loss=0.6329 train_acc_dec=0.5852 train_acc_bub=0.5049 val_loss=0.8420 val_acc_dec=0.4564 val_acc_bub=0.4328 metric_used=0.4328
[ep 100] train_loss=0.6548 train_acc_dec=0.5992 train_acc_bub=0.5166 val_loss=0.7282 val_acc_dec=0.5101 val_acc_bub=0.4701 metric_used=0.4701
[ep 125] train_loss=0.6368 train_acc_dec=0.6159 train_acc_bub=

,fold,params,train_genomes,val_genomes,test_genomes,train_bubbles,val_bubbles,test_bubbles,test_loss,test_acc_dec,test_acc_bub,test_precision_dec,test_recall_dec,test_f1_dec,test_brier,test_decisions,retry,seed
0,1/3,10569,"ecoli_variant2,ecoli_variant1",,ecoli_variant3,1211,134,334,0.697797,0.545455,0.523952,0.545706,0.994949,0.704830,0.224401,363,1,9391146
1,2/3,10569,"ecoli_variant3,ecoli_variant1",,ecoli_variant2,1319,146,214,0.750856,0.564854,0.532710,0.567227,1.000000,0.723861,0.200460,239,1,9391146
2,3/3,10569,"ecoli_variant3,ecoli_variant2",,ecoli_variant1,493,55,1131,0.691280,0.522643,0.496905,0.523455,0.998580,0.686859,0.183780,1347,1,9391146
3,1/3,10569,"ecoli_variant1,ecoli_variant3",,ecoli_variant2,1319,146,214,0.600403,0.548117,0.514019,0.546218,0.992366,0.704607,0.165324,239,2,9333662
4,2/3,10569,"ecoli_variant2,ecoli_variant3",,ecoli_variant1,493,55,1088,0.699741,0.517446,0.484375,0.517498,0.997131,0.681373,0.204716,1347,2,9333662
5,3/3,10569,"ecoli_variant2,ecoli_variant1",,ecoli_variant3,1211,134,334,0.779950,0.528926,0.514970,0.531856,1.000000,0.694394,0.196894,363,2,9333662
6,1/3,10569,"ecoli_variant2,ecoli_variant1",,ecoli_variant3,1211,134,334,0.745947,0.537190,0.505988,0.534626,0.989744,0.694245,0.181717,363,3,5017879
7,2/3,10569,"ecoli_variant3,ecoli_variant1",,ecoli_variant2,1319,146,214,0.859795,0.560669,0.542056,0.560669,1.000000,0.718499,0.197855,239,3,5017879
8,3/3,10569,"ecoli_variant3,ecoli_variant2",,ecoli_variant1,493,55,1131,0.701556,0.520416,0.502210,0.521221,0.998573,0.684932,0.193101,1347,3,5017879
9,1/3,10569,"ecoli_variant1,ecoli_variant3",,ecoli_variant2,1319,146,214,0.616254,0.577406,0.542056,0.579832,1.000000,0.734043,0.169724,239,4,6705175
